# 03 Feature Engineering — Scope and Leakage Rules

This notebook creates deterministic, leakage-free feature matrices for the MoA prediction project.

The goal is to build feature-set versions that can later be compared during model training:

- FE_V0: raw metadata + gene + cell features
- FE_V1: FE_V0 + row-wise gene/cell summary features
- FE_V2: FE_V1 + small gene-cell interaction features
- FE_V3: optional metadata interaction features

This notebook will not train models, fit scalers, fit PCA, fit quantile transformers, fit feature selectors, or use any target-derived information as input features.

All learned transformations such as scaling, PCA, quantile transformation, variance filtering, and supervised feature selection must be fitted only inside the validation/modeling pipeline.

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Project root detection
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

EXPERIMENT_LOG_DIR = PROJECT_ROOT / "outputs" / "experiment_logs"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"

DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_LOG_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_RAW_DIR:", DATA_RAW_DIR)
print("DATA_INTERIM_DIR:", DATA_INTERIM_DIR)
print("DATA_PROCESSED_DIR:", DATA_PROCESSED_DIR)
print("EXPERIMENT_LOG_DIR:", EXPERIMENT_LOG_DIR)
print("FIGURE_DIR:", FIGURE_DIR)

PROJECT_ROOT: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response
DATA_RAW_DIR: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response\data\raw
DATA_INTERIM_DIR: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response\data\interim
DATA_PROCESSED_DIR: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response\data\processed
EXPERIMENT_LOG_DIR: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response\outputs\experiment_logs
FIGURE_DIR: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response\reports\figures


In [3]:
feature_engineering_scope_table = pd.DataFrame({
    "area": [
        "Notebook purpose",
        "Allowed feature type",
        "Baseline feature versions",
        "Model training",
        "Learned transformations",
        "Target-derived features",
        "Nonscored targets",
        "Drug ID",
        "Control-row prediction rule"
    ],
    "rule": [
        "Create leakage-free deterministic feature matrices",
        "Only input-derived, row-wise features",
        "FE_V0, FE_V1, FE_V2, optional FE_V3",
        "Not performed in this notebook",
        "Not fitted in this notebook",
        "Never used as model input",
        "Never used as baseline input features",
        "Not used as baseline model input",
        "Documented here, applied only after prediction"
    ],
    "reason": [
        "Feature engineering should stay separate from modeling",
        "Safe because each feature uses only the same sample's input values",
        "Allows clean model comparison later",
        "Model training belongs to model training notebook",
        "Scaler/PCA/Quantile/selection must be fitted inside CV pipeline",
        "Would leak label information",
        "Nonscored labels are auxiliary targets only, not inputs",
        "Drug ID is useful for validation diagnostics but risky as baseline input",
        "ctl_vehicle zeroing is an inference/submission post-processing rule"
    ]
})

feature_engineering_scope_table

,area,rule,reason
0,Notebook purpose,Create leakage-free deterministic feature matr...,Feature engineering should stay separate from ...
1,Allowed feature type,"Only input-derived, row-wise features",Safe because each feature uses only the same s...
2,Baseline feature versions,"FE_V0, FE_V1, FE_V2, optional FE_V3",Allows clean model comparison later
3,Model training,Not performed in this notebook,Model training belongs to model training notebook
4,Learned transformations,Not fitted in this notebook,Scaler/PCA/Quantile/selection must be fitted i...
5,Target-derived features,Never used as model input,Would leak label information
6,Nonscored targets,Never used as baseline input features,"Nonscored labels are auxiliary targets only, n..."
7,Drug ID,Not used as baseline model input,Drug ID is useful for validation diagnostics b...
8,Control-row prediction rule,"Documented here, applied only after prediction",ctl_vehicle zeroing is an inference/submission...


In [4]:
hard_leakage_rules = pd.DataFrame({
    "rule_id": [
        "L001",
        "L002",
        "L003",
        "L004",
        "L005",
        "L006",
        "L007",
        "L008",
        "L009",
        "L010"
    ],
    "leakage_rule": [
        "No scored target column can enter X",
        "No nonscored target column can enter X",
        "No target-derived count or flag can enter X",
        "No fold-level target statistic can enter X",
        "No target co-occurrence feature can enter X",
        "No scaler is fitted before cross-validation",
        "No PCA is fitted before cross-validation",
        "No QuantileTransformer is fitted before cross-validation",
        "No feature selector is fitted before cross-validation",
        "No drug_id is used as a baseline input feature"
    ],
    "forbidden_examples": [
        "nfkb_inhibitor, proteasome_inhibitor, etc.",
        "any nonscored target label",
        "active_target_count, zero_label_flag, multi_label_count",
        "validation_positive_count, rare_label_fold_count",
        "target_pair_count, label_cluster_id",
        "StandardScaler.fit(full_train)",
        "PCA.fit(full_train)",
        "QuantileTransformer.fit(full_train)",
        "SelectKBest.fit(full_train, y)",
        "drug_id one-hot as baseline X"
    ],
    "safe_alternative": [
        "Use targets only as y",
        "Use nonscored only as advanced auxiliary y",
        "Use gene/cell input-derived summaries",
        "Use fold statistics only for diagnostics",
        "Use co-occurrence only for EDA interpretation",
        "Fit scaler inside modeling pipeline",
        "Fit PCA inside modeling pipeline",
        "Fit QuantileTransformer inside modeling pipeline",
        "Fit selector inside modeling pipeline",
        "Use drug_id only for validation-risk diagnostics"
    ]
})

hard_leakage_rules

,rule_id,leakage_rule,forbidden_examples,safe_alternative
0,L001,No scored target column can enter X,"nfkb_inhibitor, proteasome_inhibitor, etc.",Use targets only as y
1,L002,No nonscored target column can enter X,any nonscored target label,Use nonscored only as advanced auxiliary y
2,L003,No target-derived count or flag can enter X,"active_target_count, zero_label_flag, multi_la...",Use gene/cell input-derived summaries
3,L004,No fold-level target statistic can enter X,"validation_positive_count, rare_label_fold_count",Use fold statistics only for diagnostics
4,L005,No target co-occurrence feature can enter X,"target_pair_count, label_cluster_id",Use co-occurrence only for EDA interpretation
5,L006,No scaler is fitted before cross-validation,StandardScaler.fit(full_train),Fit scaler inside modeling pipeline
6,L007,No PCA is fitted before cross-validation,PCA.fit(full_train),Fit PCA inside modeling pipeline
7,L008,No QuantileTransformer is fitted before cross-...,QuantileTransformer.fit(full_train),Fit QuantileTransformer inside modeling pipeline
8,L009,No feature selector is fitted before cross-val...,"SelectKBest.fit(full_train, y)",Fit selector inside modeling pipeline
9,L010,No drug_id is used as a baseline input feature,drug_id one-hot as baseline X,Use drug_id only for validation-risk diagnostics


In [5]:
feature_category_rules = pd.DataFrame({
    "feature_category": [
        "Raw metadata",
        "Raw gene features",
        "Raw cell features",
        "Gene summary features",
        "Cell summary features",
        "Gene-cell interaction features",
        "Metadata interaction features",
        "Scaling",
        "PCA",
        "Quantile transformation",
        "Feature selection",
        "Nonscored targets",
        "Scored targets",
        "Drug ID",
        "Control-row zeroing"
    ],
    "status": [
        "allowed",
        "allowed",
        "allowed",
        "allowed",
        "allowed",
        "allowed",
        "optional",
        "pipeline-only",
        "pipeline-only",
        "pipeline-only",
        "pipeline-only",
        "forbidden-as-input",
        "forbidden-as-input",
        "diagnostic-only",
        "inference-only"
    ],
    "where_used": [
        "FE_V0 onward",
        "FE_V0 onward",
        "FE_V0 onward",
        "FE_V1 onward",
        "FE_V1 onward",
        "FE_V2 onward",
        "optional FE_V3",
        "modeling pipeline",
        "modeling pipeline",
        "modeling pipeline",
        "modeling pipeline",
        "advanced auxiliary output only",
        "y target only",
        "validation-risk analysis only",
        "submission post-processing"
    ],
    "notes": [
        "cp_type, cp_time, cp_dose",
        "g-* columns",
        "c-* columns",
        "row-wise deterministic",
        "row-wise deterministic",
        "small audited set only",
        "keep small; avoid sparse explosion",
        "fit on training folds only",
        "fit on training folds only",
        "fit on training folds only",
        "fit on training folds only",
        "never concatenate into X",
        "never concatenate into X",
        "not a baseline feature",
        "apply after prediction, not during feature creation"
    ]
})

feature_category_rules

,feature_category,status,where_used,notes
0,Raw metadata,allowed,FE_V0 onward,"cp_type, cp_time, cp_dose"
1,Raw gene features,allowed,FE_V0 onward,g-* columns
2,Raw cell features,allowed,FE_V0 onward,c-* columns
3,Gene summary features,allowed,FE_V1 onward,row-wise deterministic
4,Cell summary features,allowed,FE_V1 onward,row-wise deterministic
5,Gene-cell interaction features,allowed,FE_V2 onward,small audited set only
6,Metadata interaction features,optional,optional FE_V3,keep small; avoid sparse explosion
7,Scaling,pipeline-only,modeling pipeline,fit on training folds only
8,PCA,pipeline-only,modeling pipeline,fit on training folds only
9,Quantile transformation,pipeline-only,modeling pipeline,fit on training folds only


In [6]:
feature_engineering_scope_table.to_csv(
    EXPERIMENT_LOG_DIR / "step_1_feature_engineering_scope_table.csv",
    index=False
)

hard_leakage_rules.to_csv(
    EXPERIMENT_LOG_DIR / "step_1_hard_leakage_rules.csv",
    index=False
)

feature_category_rules.to_csv(
    EXPERIMENT_LOG_DIR / "step_1_feature_category_rules.csv",
    index=False
)

print("Step 1 scope and leakage-rule files saved successfully.")

Step 1 scope and leakage-rule files saved successfully.


In [7]:
step_1_status = {
    "step": "Step 1 — Notebook Scope and Leakage Rules",
    "status": "completed",
    "model_training_started": False,
    "learned_transform_fitted": False,
    "target_derived_features_allowed": False,
    "nonscored_targets_allowed_as_input": False,
    "drug_id_allowed_as_baseline_input": False,
    "ready_for_step_2": True,
    "next_step": "Step 2 — Load Clean Interim Data"
}

step_1_status

{'step': 'Step 1 — Notebook Scope and Leakage Rules',
 'status': 'completed',
 'model_training_started': False,
 'learned_transform_fitted': False,
 'target_derived_features_allowed': False,
 'nonscored_targets_allowed_as_input': False,
 'drug_id_allowed_as_baseline_input': False,
 'ready_for_step_2': True,
 'next_step': 'Step 2 — Load Clean Interim Data'}